In [ ]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import ssl
import certifi
import random
from datetime import datetime

import os
from dotenv import load_dotenv
from weaviate.classes.config import Property, DataType
import os
import uuid
import base64
import httpx
from datetime import datetime
from typing import List, Dict, Any, Optional, Annotated, Literal, TypedDict
import sys
sys.path.insert(0, r'..\src')

In [ ]:
load_dotenv(os.path.join("..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

def summarize_value(value: str) -> str:
    """Return masked form: ****last4 or boolean string."""
    lower = value.lower()
    if lower in ("true", "false"):
        return lower
    return "****" + value[-4:] if len(value) > 4 else "****" + value

def doublecheck_env(file_path: str):
    """Check environment variables against a .env file and print summaries."""
    if not os.path.exists(file_path):
        print(f"Did not find file {file_path}.")
        print("This is used to double check the key settings for the notebook.")
        print("This is just a check and is not required.\n")
        return

    parsed = dotenv_values(file_path)
    for key in parsed.keys():
        current = os.getenv(key)
        if current is not None:
            print(f"{key}={summarize_value(current)}")
        else:
            print(f"{key}=<not set>")
            

In [ ]:
from utils import show_prompt
from prompts_arxiv import SUMMARIZE_WEB_SEARCH
show_prompt(SUMMARIZE_WEB_SEARCH)

In [ ]:
from datetime import datetime
from sentence_transformers import SentenceTransformer
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from utils import show_prompt, stream_agent

from file_tools_arxiv import ls, read_file, write_file
from prompts_arxiv import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from research_tools_arxiv import tavily_search, think_tool, get_today_str
from state_arxiv import DeepAgentState
from task_tool_arxiv import _create_task_tool
from todo_tools_arxiv import write_todos, read_todos

import weaviate
import weaviate.classes as wvc
from sentence_transformers import SentenceTransformer
from langchain_core.messages import ToolMessage, HumanMessage
from langchain_core.tools import InjectedToolCallId, tool
from langchain_core.documents import Document
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from markdownify import markdownify
from tavily import TavilyClient
from langchain_core.messages import AnyMessage, BaseMessage
from langgraph.graph.message import add_messages
BASE_URL = "https://export.arxiv.org/api/query"

def get_arxiv_papers(
    query,
    max_results=3,
    start=0,
    mode=None,          # None → default relevance
    random_pool=100
):
    """
    Retrieve arXiv papers with extended metadata.

    Parameters:
        query (str): search keywords
        max_results (int): number of results to return
        start (int): pagination offset
        mode (str|None): 'latest', 'updated', 'random', or None (relevance)
        random_pool (int): pool size for random mode (used only for random)

    Returns:
        List[dict]: each dict contains paper metadata
    """
    import time
    time.sleep(3)
    if not query:
        raise ValueError("query must not be empty")

    search_query = query

    # Sorting logic
    if mode is None:
        sort_by = "relevance"
        sort_order = "descending"
    elif mode == "latest":
        sort_by = "submittedDate"
        sort_order = "descending"
    elif mode == "updated":
        sort_by = "lastUpdatedDate"
        sort_order = "descending"
    elif mode == "random":
        sort_by = "submittedDate"
        sort_order = "descending"
        start = random.randint(0, max(0, random_pool - max_results))
    else:
        raise ValueError("mode must be None, 'latest', 'updated', or 'random'")

    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results if mode != "random" else random_pool,
        "sortBy": sort_by,
        "sortOrder": sort_order
    }

    url = BASE_URL + "?" + urllib.parse.urlencode(params)

    # SSL context
    context = ssl.create_default_context(cafile=certifi.where())
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "ArxivAgentTool/1.0 (your_email@example.com)"}
    )

    with urllib.request.urlopen(req, context=context) as response:
        data = response.read()

    # Namespaces
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    root = ET.fromstring(data)
    papers = []

    for entry in root.findall("atom:entry", ns):
        arxiv_id = entry.find("atom:id", ns).text.split("/")[-1]
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip()
        published = entry.find("atom:published", ns).text.strip()
        updated = entry.find("atom:updated", ns).text.strip()

        # Authors
        authors = [
            author.find("atom:name", ns).text
            for author in entry.findall("atom:author", ns)
        ]

        # DOI
        doi_elem = entry.find("arxiv:doi", ns)
        doi = doi_elem.text if doi_elem is not None else None

        # Journal reference
        journal_elem = entry.find("arxiv:journal_ref", ns)
        journal_ref = journal_elem.text if journal_elem is not None else None

        # Comment
        comment_elem = entry.find("arxiv:comment", ns)
        comment = comment_elem.text if comment_elem is not None else None

        # Primary category
        primary_cat_elem = entry.find("arxiv:primary_category", ns)
        primary_category = primary_cat_elem.attrib["term"] if primary_cat_elem is not None else None

        # All categories
        categories = [cat.attrib["term"] for cat in entry.findall("atom:category", ns)]

        # Abstract URL
        abs_url = None
        for link in entry.findall("atom:link", ns):
            if link.attrib.get("rel") == "alternate":
                abs_url = link.attrib.get("href")
                break

        # PDF URL
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

        papers.append({
            "id": arxiv_id,
            "title": title,
            "authors": authors,
            "published": datetime.fromisoformat(published.replace("Z", "")),
            "updated": datetime.fromisoformat(updated.replace("Z", "")),
            "abstract": summary,
            "pdf_url": pdf_url,
            "abs_url": abs_url,
            "doi": doi,
            "journal_ref": journal_ref,
            "comment": comment,
            "primary_category": primary_category,
            "all_categories": categories
        })

    # Final random sampling
    if mode == "random":
        papers = random.sample(papers, min(max_results, len(papers)))

    return papers


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

@tool(parse_docstring=True)
def arxiv_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: int = 3
) -> Command:
    """Search arXiv for papers, store abstracts in Weaviate, retrieve top‑3, and save summaries to files.

    Args:
        query: The search query string.
        state: Injected agent state containing the current file system.
        tool_call_id: Injected tool call identifier for message responses.
        max_results

    Returns:
        Command updating the agent's files with saved paper abstracts and
        adding a tool message containing a summary of the retrieved papers.
    """
    # 1. Fetch papers from arXiv using custom function
    import time
    for attempt in range(3):          # up to 3 retries
        try:
            papers = get_arxiv_papers(query, max_results=max_results)
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(10 * (attempt + 1))  # 10s, then 20s
                continue
            return Command(
                update={
                    "messages": [
                        ToolMessage(f"arXiv search failed: {str(e)}", tool_call_id=tool_call_id)
                    ]
                }
            )

    if not papers:
        return Command(
            update={
                "messages": [
                    ToolMessage("No papers found on arXiv for this query.", tool_call_id=tool_call_id)
                ]
            }
        )

    # 2. Store in Weaviate (clear previous run)
    client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
    import time 

    try:
        if client.collections.exists("ArxivArticles"):
            client.collections.delete("ArxivArticles")
            for _ in range(10):
                if not client.collections.exists("ArxivArticles"):
                    break
                time.sleep(0.5)
    
        collection = client.collections.create(
            name="ArxivArticles",
            properties=[
                Property(name="title", data_type=DataType.TEXT),
                Property(name="abstract", data_type=DataType.TEXT),
                Property(name="arxiv_id", data_type=DataType.TEXT),
                Property(name="url", data_type=DataType.TEXT),
                Property(name="published", data_type=DataType.DATE),
            ],
            vectorizer_config=None
        )
        collection = client.collections.get("ArxivArticles")
    
        for paper in papers:
            title = paper["title"]
            abstract = paper["abstract"]
            text = title + " " + abstract
            embedding = embedding_model.encode(text).tolist()
            arxiv_id = paper["id"]
            url = paper.get("abs_url") or f"https://arxiv.org/abs/{arxiv_id}"
            
            published = paper["published"].isoformat()
            if not published.endswith('Z') and not ('+' in published or '-' in published[10:]):
                published += 'Z'
            
            collection.data.insert(
                properties={
                    "title": title,
                    "abstract": abstract,
                    "arxiv_id": arxiv_id,
                    "url": url,
                    "published": published,
                },
                vector=embedding
            )
    
        query_embedding = embedding_model.encode(query).tolist()
        response = collection.query.near_vector(
            near_vector=query_embedding,
            limit=3
        )
    finally:
        client.close()

    # 4. Save each abstract to a file and build a summary message
    files = state.get("files", {})
    saved_files = []
    summaries = []

    for i, obj in enumerate(response.objects, 1):
        props = obj.properties
        filename = f"arxiv_{props['arxiv_id']}_{uuid.uuid4().hex[:8]}.md"
        file_content = f"""# {props['title']}

**arXiv ID:** {props['arxiv_id']}
**URL:** {props['url']}
**Published:** {props['published']}

## Abstract
{props['abstract']}
"""
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {props['title']}")

    summary_text = f"""📄 Found {len(papers)} papers on arXiv for '{query}'. Retrieved top 3 by relevance:

{chr(10).join(summaries)}

Files saved: {', '.join(saved_files)}

Use read_file() to read the full abstractsW when needed."""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

## Deep Agent


* We'll give the researcher a `think_tool` and our `search_tool` above.
* We'll give our parent agent file tools, a `think_tool`, and a `task` tool. 

In [ ]:
from datetime import datetime
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from utils import show_prompt, stream_agent

from file_tools_arxiv import ls, read_file, write_file
from prompts_arxiv import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from research_tools_arxiv import tavily_search, think_tool, get_today_str
from state_arxiv import DeepAgentState
from task_tool_arxiv import _create_task_tool
from todo_tools_arxiv import write_todos, read_todos

# -------------------- Main Agent Setup --------------------
model = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)

@tool
def limited_arxiv_search(
    query: str,
    max_results: int,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Search arXiv for academic papers. Limited to 5 calls per research task."""
    if not hasattr(limited_arxiv_search.func, "call_count"):
        limited_arxiv_search.func.call_count = 0
    if limited_arxiv_search.func.call_count >= 5:
        limited_arxiv_search.func.call_count += 1 
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        "ERROR: arXiv search limit reached (5 calls).",
                        tool_call_id=tool_call_id
                    )
                ]
            }
        )
    limited_arxiv_search.func.call_count += 1

    # Call the underlying function of the original tool.
    return arxiv_search.func(
        query=query,
        state=state,
        tool_call_id=tool_call_id,
        max_results=max_results
    )


# Tools for sub‑agent (the researcher)
sub_agent_tools = [limited_arxiv_search, tavily_search, think_tool]

# Built‑in tools for main agent
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]

# Research sub‑agent definition (uses string names that match the wrapped tools)
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "prompt": RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    "tools": ["limited_arxiv_search", "tavily_search", "think_tool"],  # names must match actual tool names
}

# Create the task tool with our research sub‑agent
task_tool = _create_task_tool(
    sub_agent_tools,
    [research_sub_agent],
    model,
    DeepAgentState
)

all_tools = built_in_tools + [task_tool]

# Limits
max_concurrent_research_units = 2
max_researcher_iterations = 2
SUBAGENT_INSTRUCTIONS = SUBAGENT_USAGE_INSTRUCTIONS.format(
    max_concurrent_research_units=max_concurrent_research_units,
    max_researcher_iterations=max_researcher_iterations,
    date=datetime.now().strftime("%a %b %d, %Y").replace(" 0", " ")
)


In [ ]:
INSTRUCTIONS = (
    "# TODO MANAGEMENT\n"
    + TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# FILE SYSTEM USAGE\n"
    + FILE_USAGE_INSTRUCTIONS
    +"When you receive the researcher’s final answer, always include the list of sources they provided."
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# SUB-AGENT DELEGATION\n"
    + "Make sure to call sub-agent atleast once - in other words call think tool atleast once. \n"
    + SUBAGENT_INSTRUCTIONS
)

show_prompt(INSTRUCTIONS)

In [ ]:
# Create the main agent
main_agent = create_agent(
    model,
    all_tools,
    system_prompt=INSTRUCTIONS,
    state_schema=DeepAgentState
)

# -------------------- Helper for Random Fact --------------------
# Simple year→event mapping (can be expanded)
YEAR_EVENTS = {
    2020: "the COVID‑19 pandemic began",
    2021: "NASA's Perseverance rover landed on Mars",
    2022: "the James Webb Space Telescope released its first images",
    2023: "the AI boom continued with GPT‑4",
    2024: "the Paris Olympics were held",
}

def get_random_fact(year: int) -> str:
    if year in YEAR_EVENTS:
        return f"Fun fact: This paper was published in {year}, the year {YEAR_EVENTS[year]}."
    else:
        return f"Fun fact: This paper was published in {year}."

In [ ]:
show_prompt(RESEARCHER_INSTRUCTIONS)

In [ ]:
# We'll add the random fact after the agent finishes, by processing the final message.
# Alternatively, we could add a final step in the agent, but for simplicity we'll append it after invocation.

initial_state = {
    "messages": [{"role": "user", "content": "Supervised Learning"}],
    "todos": [],
    "files": {}
}
result = main_agent.invoke(initial_state, config={"configurable": {"thread_id": "1"}})

# Extract final answer (last message from assistant)
final_answer = result["messages"][-1].content

# Try to extract a year from any file that might contain a publication date
# For simplicity, we'll just pick a random fact from the current year if no paper found
# In a real implementation, you might parse the saved files.
#fact_year = random.choice(list(YEAR_EVENTS.keys()))  # just for demonstration
#fact = get_random_fact(fact_year)

print("\n" + "="*50)
print("FINAL ANSWER:")
print(final_answer)
#print("\n" + fact)
print("="*50)